In [2]:
import geopandas as gpd


In [5]:
# oam_foot = gpd.read_file('../oam_global_footprints.gpkg')
oam_foot

,oam_id,title,provider,acquired,gsd_cm,area_km2,center_lat,center_lon,country,tms_url,download,continent,pass_baseline,geometry
0,59e62b863d6412ef72209ae1,Carretera _ Puente Rio Chico,Santiago Pastor,2016-04-27,0.000,0.3991,-0.977484,-80.421768,,,https://oin-hotosm-temp.s3.amazonaws.com/573a1...,South America,0,"POLYGON ((-80.41892 -0.98033, -80.41892 -0.974..."
1,59e62b903d6412ef7220a022,Johnson Valley - Soggy Dry Lake Yardangs,Douglas John Ellison,2016-12-04,0.000,0.0681,34.427203,-116.595296,,,https://oin-hotosm-temp.s3.amazonaws.com/58e86...,North America,0,"POLYGON ((-116.59387 34.42603, -116.59387 34.4..."
2,59e62b903d6412ef7220a029,Johnson Valley Soggy Dry Lake Cracks,Douglas J Ellison,2016-12-04,0.000,0.1360,34.452297,-116.687835,,,https://oin-hotosm-temp.s3.amazonaws.com/58e86...,North America,0,"POLYGON ((-116.68582 34.45064, -116.68582 34.4..."
3,59e62b863d6412ef72209b3d,Tapias - Centro Poblado,UMH Unidad de Mapeo Humanitario,2016-10-01,0.000,0.3110,4.453490,-75.351012,,,https://oin-hotosm-temp.s3.amazonaws.com/57f03...,South America,0,"POLYGON ((-75.34849 4.45098, -75.34849 4.456, ..."
4,59e62b903d6412ef7220a02f,Pisgah Crater Cinder Cone,Douglas J Ellison,2016-05-01,0.000,0.8177,34.746739,-116.375186,,,https://oin-hotosm-temp.s3.amazonaws.com/58e87...,North America,0,"POLYGON ((-116.37023 34.74267, -116.37023 34.7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10671,5b948d519c75ae0006571e60,Minsk,Darafei Praliaskouski,2018-09-07,220.513,667.0820,54.060370,27.737836,,,https://oin-hotosm-temp.s3.amazonaws.com/5b948...,Europe,0,"POLYGON ((27.93606 53.94403, 27.93606 54.17671..."
10672,5b948d1b9c75ae0006571e54,Minsk,Darafei Praliaskouski,2018-09-07,368.473,70.9613,53.950881,27.527461,,,https://oin-hotosm-temp.s3.amazonaws.com/5b948...,Europe,0,"POLYGON ((27.59194 53.91294, 27.59194 53.98883..."
10673,67ce52e208baa2fabc550303,er,RGB,2024-12-09,500.228,45.2267,14.987122,43.011757,,,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,Africa,0,"POLYGON ((43.04312 14.95683, 43.04312 15.01742..."
10674,65afcfcf71b11e0001cea8fe,DTM Pesisir Poncosari,Kel PRGG,2023-09-19,517.616,4.0526,-7.988015,110.220898,,,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,Oceania,0,"POLYGON ((110.23005 -7.99708, 110.23005 -7.978..."


In [1]:
import json
import math
import requests
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, shape, mapping

# ─── CONFIGURATION ───
MAPILLARY_TOKEN = "MLY|6932788170076425|9c9b48dbda876b0e3a455489be41d7f1"
TARGET_OAM_ID = "59e62b943d6412ef7220a29d"  # Replace with any OAM ID to test
BUFFER_RADIUS_M = 35.0                      # Max distance from street to building
FOV_TOLERANCE_DEG = 45.0                    # Camera must face within ±45° of building centroid

GEOJSON_DIR = Path("../osm_geojson_results")
OAM_CACHE_FILE = Path("../cache_fast_bbox/oam/all_uav.json")

def calculate_bearing(p1, p2):
    """Calculates geographic bearing from Point 1 to Point 2."""
    dx = p2.x - p1.x
    dy = p2.y - p1.y
    angle = math.degrees(math.atan2(dx, dy))
    return (angle + 360) % 360

def angular_difference(angle1, angle2):
    """Calculates the absolute shortest distance between two angles."""
    diff = abs(angle1 - angle2) % 360
    return diff if diff <= 180 else 360 - diff

def get_scene_bbox(scene_id):
    """Retrieves the bounding box for the target scene from the OAM cache."""
    if not OAM_CACHE_FILE.exists():
        raise FileNotFoundError(f"Cannot find {OAM_CACHE_FILE}")
    
    data = json.loads(OAM_CACHE_FILE.read_text())
    for rec in data:
        if rec.get("_id") == scene_id:
            return rec.get("bbox")
    raise ValueError(f"Scene ID {scene_id} not found in catalog.")

def fetch_mapillary_data(bbox):
    """Fetches Mapillary points and sequence IDs inside the bbox."""
    min_lon, min_lat, max_lon, max_lat = bbox
    url = "https://graph.mapillary.com/images"
    
    response = requests.get(url, params={
        "access_token": MAPILLARY_TOKEN,
        "fields": "id,geometry,compass_angle,sequence_id",
        "bbox": f"{min_lon},{min_lat},{max_lon},{max_lat}",
        "limit": 1000
    })
    
    data = response.json().get("data", [])
    if not data:
        return gpd.GeoDataFrame()
        
    features = []
    for item in data:
        coords = item["geometry"]["coordinates"]
        features.append({
            "geometry": Point(coords[0], coords[1]),
            "mapillary_id": item["id"],
            "compass_angle": item.get("compass_angle", 0.0),
            "sequence_id": item.get("sequence_id", "")
        })
        
    return gpd.GeoDataFrame(features, crs="EPSG:4326")

def load_osm_buildings(scene_id):
    """Loads previously downloaded OSM buildings for the scene."""
    filepath = GEOJSON_DIR / f"{scene_id}_osm_buildings.geojson"
    if not filepath.exists():
        raise FileNotFoundError(f"OSM geometry not found: {filepath}. Run downloader first.")
    
    gdf = gpd.read_file(filepath)
    # Filter out the bounding box polygon if present
    gdf = gdf[gdf["source"] != "search_bounding_box"]
    return gdf

# ─── MAIN PROCESSING ───
print(f"🎯 Target Scene: {TARGET_OAM_ID}")


🎯 Target Scene: 59e62b943d6412ef7220a29d


In [2]:

# 1. Load Geometries
bbox = get_scene_bbox(TARGET_OAM_ID)
buildings_gdf = load_osm_buildings(TARGET_OAM_ID)
mapillary_gdf = fetch_mapillary_data(bbox)

print(f"🏢 Loaded {len(buildings_gdf)} buildings from local storage.")
print(f"📷 Fetched {len(mapillary_gdf)} Mapillary points.")

if mapillary_gdf.empty or buildings_gdf.empty:
    print("Cannot perform Line of Sight calculation: missing buildings or images.")
else:
    # 2. Convert to metric CRS (Web Mercator) for accurate distance buffers
    bldgs_m = buildings_gdf.to_crs(epsg=3857)
    maps_m = mapillary_gdf.to_crs(epsg=3857)

    # 3. Create spatial buffers around Mapillary points (simulating search radius)
    maps_buffered = maps_m.copy()
    maps_buffered["geometry"] = maps_buffered.geometry.buffer(BUFFER_RADIUS_M)

    # 4. Spatial Join: Find buildings intersecting the camera buffers
    joined = gpd.sjoin(bldgs_m, maps_buffered, how="inner", predicate="intersects")
    
    valid_links = []
    sight_lines = []

    # 5. Line of Sight (LOS) and Bearing Math
    for idx, row in joined.iterrows():
        bldg_geom = row["geometry"]  # Metric building geometry
        bldg_centroid = bldg_geom.centroid
        
        # Get original camera point in metric CRS
        cam_point = maps_m.loc[row["index_right"]].geometry
        
        # Calculate actual bearing from camera to building centroid
        bearing_to_bldg = calculate_bearing(cam_point, bldg_centroid)
        cam_compass = row["compass_angle"]
        
        # Calculate if camera is pointing at the building
        angle_diff = angular_difference(cam_compass, bearing_to_bldg)
        
        if angle_diff <= FOV_TOLERANCE_DEG:
            # Transform line back to WGS84 (EPSG:4326) for GeoJSON output
            bldg_wgs84 = buildings_gdf.loc[idx].geometry.centroid
            cam_wgs84 = mapillary_gdf.loc[row["index_right"]].geometry
            
            valid_links.append({
                "osm_id": row.get("osm_id", "unknown"),
                "mapillary_id": row["mapillary_id"],
                "distance_m": round(cam_point.distance(bldg_centroid), 1),
                "angle_diff": round(angle_diff, 1)
            })
            
            # Create a LineString ray representing the Line of Sight
            sight_lines.append({
                "type": "Feature",
                "properties": {
                    "source": "line_of_sight",
                    "osm_id": row.get("osm_id", "unknown"),
                    "mapillary_id": row["mapillary_id"]
                },
                "geometry": mapping(LineString([cam_wgs84, bldg_wgs84]))
            })

    print(f"✅ Found {len(valid_links)} valid Camera-to-Building links (within {BUFFER_RADIUS_M}m & ±{FOV_TOLERANCE_DEG}° FOV).")
    
    # 6. Generate Sequence Lines (connecting Mapillary paths)
    seq_lines = []
    for seq_id, group in mapillary_gdf.groupby("sequence_id"):
        if len(group) > 1:
            coords = [mapping(geom)["coordinates"] for geom in group.geometry]
            seq_lines.append({
                "type": "Feature",
                "properties": {"source": "mapillary_sequence", "sequence_id": seq_id},
                "geometry": {"type": "LineString", "coordinates": coords}
            })

    # 7. Compile Final GeoJSON for QGIS Inspection
    final_features = []
    
    # Add Buildings
    for _, row in buildings_gdf.iterrows():
        final_features.append({
            "type": "Feature",
            "properties": {"source": "osm_building", "osm_id": row.get("osm_id", "")},
            "geometry": mapping(row["geometry"])
        })
        
    # Add Cameras
    for _, row in mapillary_gdf.iterrows():
        final_features.append({
            "type": "Feature",
            "properties": {
                "source": "mapillary_camera", 
                "mapillary_id": row["mapillary_id"],
                "compass_angle": row["compass_angle"]
            },
            "geometry": mapping(row["geometry"])
        })

    # Add Sight Lines and Sequence Lines
    final_features.extend(sight_lines)
    final_features.extend(seq_lines)

    output_file = f"{TARGET_OAM_ID}_debug_los_filter.geojson"
    with open(output_file, "w") as f:
        json.dump({"type": "FeatureCollection", "features": final_features}, f, indent=2)

    print(f"💾 Exported visual debugging layer to: {output_file}")
    print(f"👉 Drag {output_file} into QGIS to verify the Field of View filtering.")

🏢 Loaded 98 buildings from local storage.
📷 Fetched 580 Mapillary points.
✅ Found 215 valid Camera-to-Building links (within 35.0m & ±45.0° FOV).
💾 Exported visual debugging layer to: 59e62b943d6412ef7220a29d_debug_los_filter.geojson
👉 Drag 59e62b943d6412ef7220a29d_debug_los_filter.geojson into QGIS to verify the Field of View filtering.


In [12]:
import json
import math
import requests
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point, LineString, mapping

# ─── CONFIGURATION ───
MAPILLARY_TOKEN = "MLY|6932788170076425|9c9b48dbda876b0e3a455489be41d7f1"
TARGET_OAM_ID = "676020507264d60001bd98ab"  

ROAD_BUFFER_METERS = 20.0
NEARBY_DISTANCE_METERS = 20.0  # Distance from camera to POLYGON EDGE
FOV_TOLERANCE_DEG = 50.0

GEOJSON_DIR = Path("../osm_geojson_results")
OAM_CACHE_FILE = Path("../cache_fast_bbox/oam/all_uav.json")

def get_scene_bbox(scene_id):
    data = json.loads(OAM_CACHE_FILE.read_text())
    for rec in data:
        if rec.get("_id") == scene_id:
            return rec.get("bbox")
    raise ValueError(f"Scene ID {scene_id} not found in catalog cache.")

def fetch_mapillary(bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    r = requests.get("https://graph.mapillary.com/images", params={
        "access_token": MAPILLARY_TOKEN,
        "fields": "id,geometry,compass_angle,sequence_id,is_pano",
        "bbox": f"{min_lon},{min_lat},{max_lon},{max_lat}",
        "limit": 2000
    }, timeout=30)
    data = r.json().get("data", [])
    if not data: return gpd.GeoDataFrame()
    
    features = []
    for d in data:
        # FILTER: Skip panoramic images entirely
        if d.get("is_pano") == True:
            continue
            
        features.append({
            "geometry": Point(d["geometry"]["coordinates"]), 
            "mapillary_id": d["id"], 
            "compass_angle": d.get("compass_angle", 0.0), 
            "sequence_id": d.get("sequence_id", "")
        })
    return gpd.GeoDataFrame(features, crs="EPSG:4326")

def calculate_bearing(p1, p2):
    angle = math.degrees(math.atan2(p2.x - p1.x, p2.y - p1.y))
    return (angle + 360) % 360

# ─── EXECUTION ───
print(f"🎯 Testing Polygon-Edge Distance Scoring on Scene: {TARGET_OAM_ID}")

bbox = get_scene_bbox(TARGET_OAM_ID)
bldgs_wgs84 = gpd.read_file(GEOJSON_DIR / f"{TARGET_OAM_ID}_osm_buildings.geojson")
bldgs_wgs84 = bldgs_wgs84[bldgs_wgs84["source"] != "search_bounding_box"].reset_index(drop=True)

if "osm_id" not in bldgs_wgs84.columns:
    bldgs_wgs84["osm_id"] = bldgs_wgs84.index.astype(str)

maps_wgs84 = fetch_mapillary(bbox)

total_buildings = len(bldgs_wgs84)
total_svi = len(maps_wgs84)

print(f"🏢 Total OSM Buildings: {total_buildings}")
print(f"📷 Total Non-Panoramic Mapillary Points: {total_svi}")

if total_buildings == 0 or total_svi == 0:
    print("Zero buildings or valid SVI points. Score is 0.")
else:
    utm_crs = bldgs_wgs84.estimate_utm_crs()
    bldgs_utm = bldgs_wgs84.to_crs(utm_crs)
    maps_utm = maps_wgs84.to_crs(utm_crs)

    # 1. Buffer Mapillary Paths
    seq_lines = []
    for _, group in maps_utm.groupby("sequence_id"):
        if len(group) > 1:
            seq_lines.append(LineString(group.geometry.tolist()))
        else:
            seq_lines.append(group.geometry.iloc[0])
            
    paths_buffered = gpd.GeoDataFrame(geometry=seq_lines, crs=utm_crs).buffer(ROAD_BUFFER_METERS)

    # 2. Filter Buildings on Path
    bldgs_on_path = gpd.sjoin(bldgs_utm, gpd.GeoDataFrame(geometry=paths_buffered), how="inner", predicate="intersects")
    bldgs_on_path = bldgs_on_path[~bldgs_on_path.index.duplicated(keep='first')]
    print(f"✂️ Buildings within {ROAD_BUFFER_METERS}m of driven path: {len(bldgs_on_path)}")

    # 3. True Edge-Distance & LoS Check
    valid_pairs = 0
    all_buildings_sindex = bldgs_utm.sindex
    debug_features = []

    for bldg_idx, bldg in bldgs_on_path.iterrows():
        target_pos_idx = bldgs_utm.index.get_loc(bldg_idx)
        
        # Calculate distance from ALL Mapillary points to this building's POLYGON EDGE
        distances = maps_utm.geometry.distance(bldg.geometry)
        min_dist = distances.min()
        
        # Apply strict 20m cutoff to the building facade/edge
        if min_dist > NEARBY_DISTANCE_METERS:
            continue
            
        # Get the single closest camera
        cam_idx = distances.idxmin()
        cam = maps_utm.loc[cam_idx]
        centroid = bldg.geometry.centroid
        
        # Bearing and FOV (Still aimed at centroid to ensure looking at the main building mass)
        bearing = calculate_bearing(cam.geometry, centroid)
        angle_diff = abs(cam["compass_angle"] - bearing) % 360
        angle_diff = angle_diff if angle_diff <= 180 else 360 - angle_diff
        
        cam_wgs84 = maps_wgs84.loc[cam_idx].geometry
        bldg_wgs84_cent = bldgs_wgs84.iloc[target_pos_idx].geometry.centroid
        
        if angle_diff <= FOV_TOLERANCE_DEG:
            # Occlusion Check
            los_line = LineString([cam.geometry, centroid])
            possible_blockers_pos_idx = list(all_buildings_sindex.intersection(los_line.bounds))
            
            is_occluded = False
            for blocker_pos in possible_blockers_pos_idx:
                if blocker_pos != target_pos_idx: # Safely ignore the target building itself
                    blocker_geom = bldgs_utm.iloc[blocker_pos].geometry
                    if los_line.intersects(blocker_geom):
                        is_occluded = True
                        break
            
            if not is_occluded:
                valid_pairs += 1
                status = "valid_los"
            else:
                status = "occluded"
        else:
            status = "bad_angle"

        # Add to debug visualization
        debug_features.append({
            "type": "Feature",
            "properties": {"status": status, "edge_distance_m": round(min_dist, 1), "angle_diff": round(angle_diff, 1)},
            "geometry": mapping(LineString([cam_wgs84, bldg_wgs84_cent]))
        })

    print(f"✅ Valid Non-Occluded Matches (Score): {valid_pairs}")

    # Export Debug GeoJSON
    for idx, row in bldgs_wgs84.loc[bldgs_on_path.index].iterrows():
        debug_features.append({"type": "Feature", "properties": {"source": "building_on_path", "osm_id": row.get("osm_id", "")}, "geometry": mapping(row["geometry"])})
    for _, row in maps_wgs84.iterrows():
        debug_features.append({"type": "Feature", "properties": {"source": "mapillary", "angle": row["compass_angle"]}, "geometry": mapping(row["geometry"])})

    out_file = f"{TARGET_OAM_ID}_enrichment_debug.geojson"
    with open(out_file, "w") as f:
        json.dump({"type": "FeatureCollection", "features": debug_features}, f, indent=2)
    print(f"💾 Saved debug GeoJSON to {out_file}")

🎯 Testing Polygon-Edge Distance Scoring on Scene: 676020507264d60001bd98ab
🏢 Total OSM Buildings: 181
📷 Total Non-Panoramic Mapillary Points: 836
✂️ Buildings within 20.0m of driven path: 97
✅ Valid Non-Occluded Matches (Score): 3
💾 Saved debug GeoJSON to 676020507264d60001bd98ab_enrichment_debug.geojson


In [15]:
scene  = pd.read_csv("../scene_enrichment_scores.csv")
scene['valid_los_matches'].sum()

np.int64(3452)

In [31]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import Point
from pathlib import Path

# Paths
SCORES_CSV = Path("../scene_enrichment_scores.csv")
OAM_CACHE_FILE = Path("../cache_fast_bbox/oam/all_uav.json")

print("Loading data...")
# df = pd.read_csv(SCORES_CSV)

# # 1. Get the center coordinates for all scenes from the local cache
# with open(OAM_CACHE_FILE, "r") as f:
#     catalog = json.load(f)

# points = {}
# for rec in catalog:
#     if "_id" in rec and "bbox" in rec:
#         bbox = rec["bbox"]
#         # bbox is [min_lon, min_lat, max_lon, max_lat]
#         center_lon = (bbox[0] + bbox[2]) / 2.0
#         center_lat = (bbox[1] + bbox[3]) / 2.0
#         points[str(rec["_id"])] = Point(center_lon, center_lat)

# # 2. Map the geometries to our dataframe
# df['geometry'] = df['scene_id'].astype(str).map(points)

# # Drop any rows where we couldn't find a bounding box, just in case
# df = df.dropna(subset=['geometry'])

# # Convert to a GeoDataFrame
# gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# # 3. Load GeoPandas' built-in World Map
# print("Performing spatial join with global map...")
# world = gpd.read_file('../NE/ne_110m_admin_0_map_units.shp')

# 4. Perform Spatial Join (Find which country polygon intersects our center point)
gdf_mapped = gpd.sjoin(gdf, world[['geometry', 'REGION_UN','SUBREGION','ADMIN']], how="left", predicate="intersects")

# Rename 'name' to 'country' for clarity
gdf_mapped = gdf_mapped.rename(columns={'name': 'country'})

# 5. Clean up and save back to CSV
# Drop the spatial join artifact columns so it remains a clean, standard CSV
final_df = pd.DataFrame(gdf_mapped.drop(columns=['geometry', 'index_right']))

final_df.to_csv(SCORES_CSV, index=False)

print(f"✅ Successfully added 'country' and 'continent' to {SCORES_CSV.name}!")
print("\nPreview of the enriched dataset:")
# print(final_df[['scene_id', 'valid_los_matches', 'country', 'continent']].head(10))

Loading data...
✅ Successfully added 'country' and 'continent' to scene_enrichment_scores.csv!

Preview of the enriched dataset:


In [73]:
final_nonzero = final_df[final_df['valid_los_matches'] != 0]
final_nonzero

,scene_id,title,gsd_cm,area_km2,mapillary_ok,osm_ok,building_count,pass,reason,elapsed_s,total_buildings,total_svi_points,buildings_on_path,valid_los_matches,REGION_UN,SUBREGION,ADMIN
0,68b709ea5288a43ff3e919db,Freetown_New_England_and_Kingtom_COG,4.000,17.4460,True,True,24,True,pass,0.00,69292,810,21030,277,NaN,NaN,NaN
1,688ca9be5c13f65e3cc03bcc,Kroo Bay KYC - Full,4.988,1.4478,True,True,2,True,pass,0.01,8898,876,8042,240,NaN,NaN,NaN
2,686f12ef0af31ae7caac8152,Kroo_Bay_Disability_Analysis,5.000,1.4563,True,True,2,True,pass,0.01,8962,872,6529,239,NaN,NaN,NaN
3,61195da8ac1a340005a1cd6c,"вул. Данила Нечая, Вінниця",4.158,5.3168,True,False,0,False,no_residential_osm,0.01,3832,1359,2503,186,Europe,Eastern Europe,Ukraine
4,686eb4010af31ae7caac165f,Kolleh_Town_Disability_Analysis,5.000,1.4661,True,False,0,False,no_residential_osm,0.01,5876,651,5262,185,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,69bb03dea2cd858f4ba47ee3,Mathew 2016 -Jérémie,5.746,8.3752,True,False,0,False,no_residential_osm,0.00,12989,1,1,1,NaN,NaN,NaN
145,5e314797fbfe8c00073c70e0,2020-01-10 台中梧棲龍井安良排水治理工程(2-4K)提案階段 (Anlian Ch...,3.029,3.3585,True,False,0,False,no_residential_osm,0.01,6,1173,6,1,Asia,Eastern Asia,Taiwan
146,6420eb3b1a8878000512122b,Rangitopuni_230324_Ortho_0050m,5.000,9.0059,True,False,0,False,no_residential_osm,0.00,301,196,63,1,Oceania,Australia and New Zealand,New Zealand
147,5ec3c8fdaff5010005865c51,"СПб, Невский район, ул. Седова, Epoch 2010.0",5.422,2.3030,True,False,0,False,no_residential_osm,0.00,571,15,27,1,Europe,Eastern Europe,Russia


In [81]:
print(['valid_los_matches'].sum())

3452


In [ ]:
all_uav = pd.read_json('../cache_fast_bbox/oam/all_uav.json')


,_id,uuid,__v,title,projection,bbox,footprint,gsd,file_size,acquisition_start,acquisition_end,platform,provider,contact,properties,uploaded_at,meta_uri,geojson,user
0,59e62b863d6412ef72209ae1,https://oin-hotosm-temp.s3.amazonaws.com/573a1...,0,Carretera _ Puente Rio Chico,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS...","[-80.4248105, -0.9801461999990001, -80.4187253...","POLYGON((-80.4248105 -0.9748216999999999,-80.4...",1.857046e-07,47649982,2016-04-27T05:00:00.000Z,2016-04-28T17:00:03.000Z,uav,Santiago Pastor,"fruizc,francisco.ruiz@engeotec.net",{'wmts': 'http://tiles.openaerialmap.org/573a1...,2016-05-16 00:00:00+00:00,https://oin-hotosm-temp.s3.amazonaws.com/573a1...,"{'bbox': [-80.4248105, -0.9801461999990001, -8...",NaN
1,59e62b903d6412ef7220a022,https://oin-hotosm-temp.s3.amazonaws.com/58e86...,0,Johnson Valley - Soggy Dry Lake Yardangs,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS...","[-116.59716988178567, 34.42630988440853, -116....",POLYGON((-116.59716988178567 34.42809659794853...,1.989375e-07,32572534,2016-12-04T20:00:00.000Z,2016-12-04T21:00:00.000Z,uav,Douglas John Ellison,"Doug Ellison,douglas.j.ellison@gmail.com","{'license': 'CC-BY 4.0', 'sensor': 'DJI Phanto...",2017-04-08 04:57:38.054000+00:00,https://oin-hotosm-temp.s3.amazonaws.com/58e86...,"{'bbox': [-116.59716988178567, 34.426309884408...",NaN
2,59e62b903d6412ef7220a029,https://oin-hotosm-temp.s3.amazonaws.com/58e86...,0,Johnson Valley Soggy Dry Lake Cracks,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS...","[-116.68984993967797, 34.4506361486674, -116.6...",POLYGON((-116.68984993967797 34.45395858684593...,1.989696e-07,38470684,2016-12-04T23:00:00.000Z,2016-12-05T00:00:00.000Z,uav,Douglas J Ellison,"Doug Ellison,douglas.j.ellison@gmail.com","{'license': 'CC-BY 4.0', 'sensor': 'DJI Phanto...",2017-04-08 04:59:05.713000+00:00,https://oin-hotosm-temp.s3.amazonaws.com/58e86...,"{'bbox': [-116.68984993967797, 34.450636148667...",NaN
3,59e62b863d6412ef72209b3d,https://oin-hotosm-temp.s3.amazonaws.com/57f03...,0,Tapias - Centro Poblado,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS...","[-75.35279172145262, 4.449932472396174, -75.34...",POLYGON((-75.35279172145262 4.4570479457401735...,2.928145e-07,33151045,2016-10-01T05:00:00.000Z,2016-10-01T22:10:07.345Z,uav,UMH Unidad de Mapeo Humanitario,"Humano,fredyrivera@gmail.com",{'wmts': 'http://tiles.openaerialmap.org/57f03...,2016-10-01 00:00:00+00:00,https://oin-hotosm-temp.s3.amazonaws.com/57f03...,"{'bbox': [-75.35279172145262, 4.44993247239617...",NaN
4,59e62b903d6412ef7220a02f,https://oin-hotosm-temp.s3.amazonaws.com/58e87...,0,Pisgah Crater Cinder Cone,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS...","[-116.38018350872743, 34.74269850450097, -116....",POLYGON((-116.38018350872743 34.75077985936303...,2.990238e-07,53554806,2016-05-01T07:00:00.000Z,2016-05-01T07:00:00.000Z,uav,Douglas J Ellison,"Doug Ellison,douglas.j.ellison@gmail.com","{'license': 'CC-BY 4.0', 'sensor': 'Phantom 3 ...",2017-04-08 05:11:08.440000+00:00,https://oin-hotosm-temp.s3.amazonaws.com/58e87...,"{'bbox': [-116.38018350872743, 34.742698504500...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11130,5b948d519c75ae0006571e60,https://oin-hotosm-temp.s3.amazonaws.com/5b948...,0,Minsk,"PROJCS[""unnamed"",GEOGCS[""unnamed ellipse"",DATU...","[27.561734, 53.929415, 27.913937, 54.191325]","POLYGON ((27.561734 53.929415, 27.913937 53.92...",2.205126e+00,7050553,2018-09-07T21:00:00.000Z,2018-09-07T22:00:00.000Z,uav,Darafei Praliaskouski,"Darafei Praliaskouski,komzpa@gmail.com",{'wmts': 'https://tiles.openaerialmap.org/5b94...,2018-09-09 03:02:41.934000+00:00,https://oin-hotosm-temp.s3.amazonaws.com/5b948...,"{'bbox': [27.561734, 53.929415, 27.913937, 54....","{'_id': '5a02e4c331eff4000c380594', 'name': 'D..."
11131,5b948d1b9c75ae0006571e54,https://oin-hotosm-temp.s3.amazonaws.com/5b948...,0,Minsk,"PROJCS[""unnamed"",GEOGCS[""unnamed ellipse"",DATU...","[27.451793, 53.918546,

In [142]:
all_uav['geometry'] = all_uav['geojson'].apply(shape)
all_uav = gpd.GeoDataFrame(
    all_uav,
    geometry='geometry',
    crs='EPSG:4326'
)
# Save as GeoPackage
out = '../cache_fast_bbox/oam/all_uav.gpkg'

all_uav.to_file(
    out,
    driver='GPKG'
)


In [162]:
scene_enrichment_scores = pd.read_csv('../scene_enrichment_scores.csv')
# Match all_uav._id -> scene_enrichment_scores.scene_id
matched = all_uav.merge(
    scene_enrichment_scores,
    left_on='_id',
    right_on='scene_id',
    how='inner',
    suffixes=('_uav', '_los')
)

# Convert GeoJSON dict to Shapely geometry
matched['geometry'] = matched['geojson'].apply(shape)

# Create GeoDataFrame
matched_gdf = gpd.GeoDataFrame(
    matched,
    geometry='geometry',
    crs='EPSG:4326'
)
matched_gdf.drop(columns=['__v', 'scene_id','mapillary_ok', 'osm_ok', 'building_count', 'pass','reason', 'elapsed_s','title_los'], inplace=True)

matched_gdf.rename(columns={'title_uav': 'title'}, inplace=True)
matched_gdf
# # Save as GeoPackage
# out = '../cache_fast_bbox/oam/final_nonzero_uav2.gpkg'

# matched_gdf.to_file(
#     out,
#     layer='uav_matches',
#     driver='GPKG'
# )

# print(f"Saved {len(matched_gdf)} records to {out}") 

,_id,uuid,title,projection,bbox,footprint,gsd,file_size,acquisition_start,acquisition_end,...,meta_uri,geojson,user,geometry,gsd_cm,area_km2,total_buildings,total_svi_points,buildings_on_path,valid_los_matches
0,6a2d426391730de10f6bb33e,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,The Original City Center of Jeju City in Korea,"\nPROJCRS[""KGD2002 / Central Belt 2010"",\n ...","[126.516331, 33.506763, 126.533065, 33.518614]","POLYGON ((126.516331 33.506763, 126.533065 33....",0.010000,3231859293,2023-09-30T15:00:00.000Z,2023-10-31T14:59:00.000Z,...,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,"{'type': 'MultiPolygon', 'coordinates': [[[[12...","{'_id': '6a2a3dc4882f78eab98373e0', 'name': 'S...","MULTIPOLYGON (((126.51633 33.51855, 126.5164 3...",1.000,2.0372,4908,471,2954,40
1,652a6a10db601900014daad6,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,M2_22Feb22_35_transparent_mosaic_group1,"\nPROJCRS[""WGS 84 / UTM zone 46N"",\n BASEGE...","[90.360887, 23.837113, 90.3739, 23.846721]","POLYGON ((90.360887 23.837113, 90.3739 23.8371...",0.012120,1032502531,2023-10-13T18:00:00.000Z,2023-10-14T04:10:40.342Z,...,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,"{'bbox': [90.360887, 23.837113, 90.3739, 23.84...","{'_id': '64a3a9e664adbc00012e07f9', 'name': 'd...","MULTIPOLYGON (((90.36089 23.8465, 90.36108 23....",1.212,1.4090,1137,1877,889,30
2,6529bc2ddb601900014daac4,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,sagufta_03Feb22_M1_transparent_mosaic_group1_33,"\nPROJCRS[""WGS 84 / UTM zone 46N"",\n BASEGE...","[90.374979, 23.829543, 90.384209, 23.842295]","POLYGON ((90.374979 23.829543, 90.384209 23.82...",0.012260,987138849,2023-10-12T18:00:00.000Z,2023-10-12T22:38:10.456Z,...,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,"{'bbox': [90.374979, 23.829543, 90.384209, 23....","{'_id': '64a3a9e664adbc00012e07f9', 'name': 'd...","MULTIPOLYGON (((90.38021 23.84053, 90.38021 23...",1.226,1.3265,1110,1665,558,11
3,5a9f2f195a9ef7cb5da703e4,https://oin-hotosm-temp.s3.amazonaws.com/5a9f2...,TC-Gita-2018_UAV4R_Flight8,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS...","[-175.2531898897765, -21.204050291607338, -175...","POLYGON ((-175.253189889777 -21.2040502916073,...",0.014327,891845475,2018-02-24T11:00:00.000Z,2018-02-24T11:00:00.000Z,...,https://oin-hotosm-temp.s3.amazonaws.com/5a9f2...,"{'bbox': [-175.2531898897765, -21.204050291607...","{'_id': '5a0d77ff31eff4000c3805da', 'name': 'C...","POLYGON ((-175.25319 -21.20405, -175.22681 -21...",1.433,29.7513,3979,66,351,18
4,5efb1ead8fec2a0005992d10,https://oin-hotosm-temp.s3.amazonaws.com/5efb1...,Ile de Saint Louis,"PROJCS[""WGS 84 / UTM zone 28N"",GEOGCS[""WGS 84""...","[-16.507769, 16.018057, -16.500973, 16.039249]","POLYGON ((-16.507769 16.018057, -16.500973 16....",0.014407,754731543,2019-08-17T22:00:00.000Z,2019-08-17T23:00:00.000Z,...,https://oin-hotosm-temp.s3.amazonaws.com/5efb1...,"{'bbox': [-16.507769, 16.018057, -16.500973, 1...","{'_id': '5eec7f22d6806200078e07f7', 'name': 'L...","MULTIPOLYGON (((-16.50777 16.0392, -16.50761 1...",1.441,1.7055,2511,1782,2055,59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
712,60f45e8a42311e0006681a1d,https://oin-hotosm-temp.s3.amazonaws.com/60f45...,Kar'er 1,"PROJCS[""WGS 84 / UTM zone 35N"",GEOGCS[""WGS 84""...","[27.15347, 54.113759, 27.177924, 54.126002]","POLYGON ((27.15347 54.113759, 27.177924 54.113...",0.059693,117417305,2021-07-15T21:00:00.000Z,2021-07-15T22:00:00.000Z,...,https://oin-hotosm-temp.s3.amazonaws.com/60f45...,"{'bbox': [27.15347, 54.113759, 27.177924, 54.1...","{'_id': '5d4b128aff91220007fbc33c', 'name': 'А...","MULTIPOLYGON (((27.15352 54.126, 27.15347 54.1...",5.969,2.1620,111,1451,2,1
713,64b01515f1006f000147ae78,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,Kalobeyei_Mission 6_Flight_02,"\nPROJCRS[""WGS 84 / UTM zone 36N"",\n BASEGE...","[34.731548, 3.735658, 34.75056, 3.749924]","POLYGON ((34.731548 3.735658, 34.75056 3.73565...",0.059908,49043703,

In [163]:
matched_gdf["has_overlap"] = False

for i, geom in matched_gdf.geometry.items():
    others = matched_gdf.geometry.drop(index=i)
    matched_gdf.loc[i, "has_overlap"] = others.overlaps(geom).any()
matched_gdf

,_id,uuid,title,projection,bbox,footprint,gsd,file_size,acquisition_start,acquisition_end,...,geojson,user,geometry,gsd_cm,area_km2,total_buildings,total_svi_points,buildings_on_path,valid_los_matches,has_overlap
0,6a2d426391730de10f6bb33e,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,The Original City Center of Jeju City in Korea,"\nPROJCRS[""KGD2002 / Central Belt 2010"",\n ...","[126.516331, 33.506763, 126.533065, 33.518614]","POLYGON ((126.516331 33.506763, 126.533065 33....",0.010000,3231859293,2023-09-30T15:00:00.000Z,2023-10-31T14:59:00.000Z,...,"{'type': 'MultiPolygon', 'coordinates': [[[[12...","{'_id': '6a2a3dc4882f78eab98373e0', 'name': 'S...","MULTIPOLYGON (((126.51633 33.51855, 126.5164 3...",1.000,2.0372,4908,471,2954,40,False
1,652a6a10db601900014daad6,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,M2_22Feb22_35_transparent_mosaic_group1,"\nPROJCRS[""WGS 84 / UTM zone 46N"",\n BASEGE...","[90.360887, 23.837113, 90.3739, 23.846721]","POLYGON ((90.360887 23.837113, 90.3739 23.8371...",0.012120,1032502531,2023-10-13T18:00:00.000Z,2023-10-14T04:10:40.342Z,...,"{'bbox': [90.360887, 23.837113, 90.3739, 23.84...","{'_id': '64a3a9e664adbc00012e07f9', 'name': 'd...","MULTIPOLYGON (((90.36089 23.8465, 90.36108 23....",1.212,1.4090,1137,1877,889,30,False
2,6529bc2ddb601900014daac4,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,sagufta_03Feb22_M1_transparent_mosaic_group1_33,"\nPROJCRS[""WGS 84 / UTM zone 46N"",\n BASEGE...","[90.374979, 23.829543, 90.384209, 23.842295]","POLYGON ((90.374979 23.829543, 90.384209 23.82...",0.012260,987138849,2023-10-12T18:00:00.000Z,2023-10-12T22:38:10.456Z,...,"{'bbox': [90.374979, 23.829543, 90.384209, 23....","{'_id': '64a3a9e664adbc00012e07f9', 'name': 'd...","MULTIPOLYGON (((90.38021 23.84053, 90.38021 23...",1.226,1.3265,1110,1665,558,11,False
3,5a9f2f195a9ef7cb5da703e4,https://oin-hotosm-temp.s3.amazonaws.com/5a9f2...,TC-Gita-2018_UAV4R_Flight8,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS...","[-175.2531898897765, -21.204050291607338, -175...","POLYGON ((-175.253189889777 -21.2040502916073,...",0.014327,891845475,2018-02-24T11:00:00.000Z,2018-02-24T11:00:00.000Z,...,"{'bbox': [-175.2531898897765, -21.204050291607...","{'_id': '5a0d77ff31eff4000c3805da', 'name': 'C...","POLYGON ((-175.25319 -21.20405, -175.22681 -21...",1.433,29.7513,3979,66,351,18,True
4,5efb1ead8fec2a0005992d10,https://oin-hotosm-temp.s3.amazonaws.com/5efb1...,Ile de Saint Louis,"PROJCS[""WGS 84 / UTM zone 28N"",GEOGCS[""WGS 84""...","[-16.507769, 16.018057, -16.500973, 16.039249]","POLYGON ((-16.507769 16.018057, -16.500973 16....",0.014407,754731543,2019-08-17T22:00:00.000Z,2019-08-17T23:00:00.000Z,...,"{'bbox': [-16.507769, 16.018057, -16.500973, 1...","{'_id': '5eec7f22d6806200078e07f7', 'name': 'L...","MULTIPOLYGON (((-16.50777 16.0392, -16.50761 1...",1.441,1.7055,2511,1782,2055,59,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
712,60f45e8a42311e0006681a1d,https://oin-hotosm-temp.s3.amazonaws.com/60f45...,Kar'er 1,"PROJCS[""WGS 84 / UTM zone 35N"",GEOGCS[""WGS 84""...","[27.15347, 54.113759, 27.177924, 54.126002]","POLYGON ((27.15347 54.113759, 27.177924 54.113...",0.059693,117417305,2021-07-15T21:00:00.000Z,2021-07-15T22:00:00.000Z,...,"{'bbox': [27.15347, 54.113759, 27.177924, 54.1...","{'_id': '5d4b128aff91220007fbc33c', 'name': 'А...","MULTIPOLYGON (((27.15352 54.126, 27.15347 54.1...",5.969,2.1620,111,1451,2,1,False
713,64b01515f1006f000147ae78,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,Kalobeyei_Mission 6_Flight_02,"\nPROJCRS[""WGS 84 / UTM zone 36N"",\n BASEGE...","[34.731548, 3.735658, 34.75056, 3.749924]","POLYGON ((34.731548 3.735658, 34.75056 3.73565...",0.059908,49043703,2022-11-10T06:00:00.000Z,2022-11-10T09:00:00.000Z,...,"{'bbox': [34.731548, 3.735658, 34.75056, 3.749...","{'_id': '64b00099f1006f000147ae53', 'name': 'W...","MULTIPOLYGON (((34.73158 3.74992, 34.73155 3.7...",5.991,3.3346,633,727,581,2,True
714,5f133b8dbaa90f00

In [164]:
matched_gdf.to_file('../los_matches.gpkg')

In [ ]:

# Natural Earth
world_path = gpd.datasets.get_path("naturalearth_lowres")
world = gpd.read_file(world_path)

# Same CRS
matched_gdf = matched_gdf.to_crs(world.crs)

Keep country + continent + geometry
world_lookup = world[["ADMIN", "CONTINENT", "geometry"]].copy()

# Spatial intersection
matched_gdf = gpd.sjoin(
    matched_gdf,
    world_lookup,
    how="left",
    predicate="intersects"
)

# Rename columns
matched_gdf = matched_gdf.rename(columns={
    "ADMIN": "country",
    "CONTINENT": "continent"
})

# Remove join index
matched_gdf = matched_gdf.drop(columns=["index_right"], errors="ignore")
import pandas as pd
import geopandas as gpd

print(matched_gdf['continent'].isna().sum())
missing_mask = (
    matched_gdf["country"].isna() |
    matched_gdf["continent"].isna()
)

missing = matched_gdf.loc[missing_mask].copy()

print("Missing before:", len(missing))

# Project to meters for nearest-distance calculation
missing_projected = missing.to_crs("EPSG:6933")
world_projected = world_lookup.to_crs("EPSG:6933")

nearest = gpd.sjoin_nearest(
    missing_projected,
    world_projected[["ADMIN", "CONTINENT", "geometry"]],
    how="left",
    distance_col="match_distance"
)

# ONLY update missing values
for idx in missing.index:

    if pd.isna(matched_gdf.loc[idx, "country"]):
        matched_gdf.loc[idx, "country"] = nearest.loc[idx, "ADMIN"]

    if pd.isna(matched_gdf.loc[idx, "continent"]):
        matched_gdf.loc[idx, "continent"] = nearest.loc[idx, "CONTINENT"]


# Check result
print("Missing country:", matched_gdf["country"].isna().sum())
print("Missing continent:", matched_gdf["continent"].isna().sum())


In [185]:
non_zero = matched_gdf[matched_gdf['valid_los_matches'] != 0]
non_zero['country'].value_counts()

country
Bangladesh                     55
Russia                         48
Haiti                          29
Philippines                    27
Bolivia                        25
Japan                          24
Kenya                          22
Puerto Rico                    21
Mozambique                     17
Ukraine                        16
Montenegro                     13
Taiwan                         13
Sierra Leone                   12
Liberia                        12
Trinidad and Tobago             9
United States of America        8
Canada                          7
Brazil                          7
Georgia                         7
Colombia                        7
Paraguay                        6
Gabon                           5
Indonesia                       5
Argentina                       4
Iceland                         4
The Bahamas                     4
Ghana                           4
Ivory Coast                     3
Mexico                          3
United

In [ ]:

# Match all_uav._id -> final_nonzero.scene_id
matched = all_uav.merge(
    final_nonzero,
    left_on='_id',
    right_on='scene_id',
    how='inner',
    suffixes=('_uav', '_los')
)

# Convert GeoJSON dict to Shapely geometry
matched['geometry'] = matched['geojson'].apply(shape)

# Create GeoDataFrame
matched_gdf = gpd.GeoDataFrame(
    matched,
    geometry='geometry',
    crs='EPSG:4326'
)

# Save as GeoPackage
out = '../cache_fast_bbox/oam/final_nonzero_uav2.gpkg'

matched_gdf.to_file(
    out,
    layer='uav_matches',
    driver='GPKG'
)

print(f"Saved {len(matched_gdf)} records to {out}")

In [186]:
# matched_gdf = gpd.read_file('../pre_dataset.gpkg')
matched_gdf.head()

,_id,uuid,title,projection,bbox,footprint,gsd,file_size,acquisition_start,acquisition_end,...,geometry,gsd_cm,area_km2,total_buildings,total_svi_points,buildings_on_path,valid_los_matches,has_overlap,country,continent
0,6a2d426391730de10f6bb33e,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,The Original City Center of Jeju City in Korea,"\nPROJCRS[""KGD2002 / Central Belt 2010"",\n ...","[126.516331, 33.506763, 126.533065, 33.518614]","POLYGON ((126.516331 33.506763, 126.533065 33....",0.010000,3231859293,2023-09-30T15:00:00.000Z,2023-10-31T14:59:00.000Z,...,"MULTIPOLYGON (((126.51633 33.51855, 126.5164 3...",1.000,2.0372,4908,471,2954,40,False,South Korea,Asia
1,652a6a10db601900014daad6,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,M2_22Feb22_35_transparent_mosaic_group1,"\nPROJCRS[""WGS 84 / UTM zone 46N"",\n BASEGE...","[90.360887, 23.837113, 90.3739, 23.846721]","POLYGON ((90.360887 23.837113, 90.3739 23.8371...",0.012120,1032502531,2023-10-13T18:00:00.000Z,2023-10-14T04:10:40.342Z,...,"MULTIPOLYGON (((90.36089 23.8465, 90.36108 23....",1.212,1.4090,1137,1877,889,30,False,Bangladesh,Asia
2,6529bc2ddb601900014daac4,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,sagufta_03Feb22_M1_transparent_mosaic_group1_33,"\nPROJCRS[""WGS 84 / UTM zone 46N"",\n BASEGE...","[90.374979, 23.829543, 90.384209, 23.842295]","POLYGON ((90.374979 23.829543, 90.384209 23.82...",0.012260,987138849,2023-10-12T18:00:00.000Z,2023-10-12T22:38:10.456Z,...,"MULTIPOLYGON (((90.38021 23.84053, 90.38021 23...",1.226,1.3265,1110,1665,558,11,False,Bangladesh,Asia
3,5a9f2f195a9ef7cb5da703e4,https://oin-hotosm-temp.s3.amazonaws.com/5a9f2...,TC-Gita-2018_UAV4R_Flight8,"GEOGCS[""WGS 84"",DATUM[""WGS_1984"",SPHEROID[""WGS...","[-175.2531898897765, -21.204050291607338, -175...","POLYGON ((-175.253189889777 -21.2040502916073,...",0.014327,891845475,2018-02-24T11:00:00.000Z,2018-02-24T11:00:00.000Z,...,"POLYGON ((-175.25319 -21.20405, -175.22681 -21...",1.433,29.7513,3979,66,351,18,True,Fiji,Oceania
4,5efb1ead8fec2a0005992d10,https://oin-hotosm-temp.s3.amazonaws.com/5efb1...,Ile de Saint Louis,"PROJCS[""WGS 84 / UTM zone 28N"",GEOGCS[""WGS 84""...","[-16.507769, 16.018057, -16.500973, 16.039249]","POLYGON ((-16.507769 16.018057, -16.500973 16....",0.014407,754731543,2019-08-17T22:00:00.000Z,2019-08-17T23:00:00.000Z,...,"MULTIPOLYGON (((-16.50777 16.0392, -16.50761 1...",1.441,1.7055,2511,1782,2055,59,False,Senegal,Africa


In [188]:
import geopandas as gpd
import pandas as pd
import networkx as nx
from pathlib import Path

# ─── CONFIGURATION ───
ISLAND_MATCH_CAP = 250  # Maximum matches we will keep from any single overlapping cluster

# # ─── 1. Load Data & Hardcode Missing Continents ───
# matched_gdf = gpd.read_file('../pre_dataset.gpkg')

if '_id' in matched_gdf.columns and 'scene_id' not in matched_gdf.columns:
    matched_gdf['scene_id'] = matched_gdf['_id'].astype(str)
else:
    matched_gdf['scene_id'] = matched_gdf['scene_id'].astype(str)

country_col = 'country' if 'country' in matched_gdf.columns else 'NAME_2'

matched_gdf['valid_los_matches'] = matched_gdf['valid_los_matches'].fillna(0).astype(int)

# ─── 2. Build Overlap Graph (Islands) ───
overlaps = gpd.sjoin(
    matched_gdf[['scene_id', 'geometry']], 
    matched_gdf[['scene_id', 'geometry']], 
    how='inner', predicate='intersects'
)

G = nx.Graph()
for sid in matched_gdf['scene_id']:
    G.add_node(sid)

for _, row in overlaps.iterrows():
    s1 = row['scene_id_left']
    s2 = row['scene_id_right']
    if s1 != s2:
        G.add_edge(s1, s2)

clusters = list(nx.connected_components(G))

# ─── 3. Aggregate Island Statistics & CAP MASSIVE CLUSTERS ───
islands = []
discarded_scenes = 0

for idx, cluster in enumerate(clusters):
    cluster_scenes = list(cluster)
    # Get all scenes in this cluster, sorted by highest match count first
    sub = matched_gdf[matched_gdf['scene_id'].isin(cluster_scenes)].sort_values(by='valid_los_matches', ascending=False)
    
    kept_scenes = []
    running_sum = 0
    
    for _, row in sub.iterrows():
        # If we already have enough matches from this overlapping cluster, discard the rest
        if running_sum >= ISLAND_MATCH_CAP and len(kept_scenes) > 0:
            discarded_scenes += 1
            continue
            
        kept_scenes.append(row['scene_id'])
        running_sum += row['valid_los_matches']
    
    # Filter 'sub' to only the scenes we decided to keep
    sub = sub[sub['scene_id'].isin(kept_scenes)]
    
    if running_sum > 0:
        islands.append({
            'island_id': idx,
            'scenes': kept_scenes,
            'scene_count': len(kept_scenes),
            'total_matches': int(sub['valid_los_matches'].sum()),
            'continents': list(sub['continent'].dropna().unique()),
            'countries': list(sub[country_col].dropna().unique()),
            'primary_continent': sub['continent'].mode().iloc[0] if not sub['continent'].empty else 'Unknown'
        })

df_islands = pd.DataFrame(islands)
print(f"Discarded {discarded_scenes} heavily overlapping scenes to enforce geographic diversity.")

# ─── 4. Continent-Stratified Bucket Allocation ───
buckets = {i: {'scenes': [], 'total_matches': 0, 'countries': set(), 'continents': set()} for i in range(1, 11)}

# Iterate through each continent one by one to ensure fair distribution
for continent in df_islands['primary_continent'].unique():
    # Sort the islands in this continent from largest to smallest
    cont_islands = df_islands[df_islands['primary_continent'] == continent].sort_values(by='total_matches', ascending=False)
    
    for _, island in cont_islands.iterrows():
        # Always give the next island to the bucket with the currently lowest total matches
        target_bucket = min(buckets.keys(), key=lambda b: buckets[b]['total_matches'])
        
        buckets[target_bucket]['scenes'].extend(island['scenes'])
        buckets[target_bucket]['total_matches'] += island['total_matches']
        buckets[target_bucket]['countries'].update(island['countries'])
        buckets[target_bucket]['continents'].update(island['continents'])

# ─── 5. Summary & Save Master Assignments ───
print("\n" + "=" * 80)
print("10 STUDENT BUCKET ALLOCATION SUMMARY (Stratified by Continent)")
print("=" * 80)

assignment_records = []
for b_id, b_data in sorted(buckets.items()):
    print(f"Bucket {b_id:02d}: Saved {len(b_data['scenes']):02d} scenes ({b_data['total_matches']} potential matches) -> student_{b_id:02d}_scenes.gpkg")
    print(f"           Continents: {', '.join(sorted(b_data['continents']))}")
    print(f"           Countries:  {', '.join(sorted(b_data['countries']))}")
    print("-" * 80)
    
    for sid in b_data['scenes']:
        assignment_records.append({'scene_id': sid, 'bucket_id': b_id})

df_assign = pd.DataFrame(assignment_records)
matched_gdf = matched_gdf.drop(columns=['bucket_id'], errors='ignore')
# Filter master matched_gdf to ONLY the scenes we assigned (drops the discarded overlaps)
matched_gdf = matched_gdf.merge(df_assign, on='scene_id', how='inner')

matched_gdf.to_file('../pre_dataset.gpkg', driver='GPKG')
matched_gdf.drop(columns=['geometry']).to_csv('../student_scene_assignments.csv', index=False)

# ─── 6. Export Individual Student GeoPackages ───
out_dir = Path('../student_packages')
out_dir.mkdir(parents=True, exist_ok=True)

for b_id in range(1, 11):
    student_gdf = matched_gdf[matched_gdf['bucket_id'] == b_id].copy()
    # columns_to_keep = ['scene_id', 'title', 'acquisition_start', 'gsd', 
    #                    'area_km2', 'total_buildings', 'total_svi_points', 'has_overlap',
    #                    'valid_los_matches', 'country', 'continent', 'geometry']
    
    # existing_columns = [c for c in columns_to_keep if c in student_gdf.columns]
    # student_gdf = student_gdf[existing_columns]
    
    out_file = out_dir / f'student_{b_id:02d}_scenes.gpkg'
    student_gdf.to_file(out_file, driver='GPKG')

Discarded 0 heavily overlapping scenes to enforce geographic diversity.

10 STUDENT BUCKET ALLOCATION SUMMARY (Stratified by Continent)
Bucket 01: Saved 18 scenes (911 potential matches) -> student_01_scenes.gpkg
           Continents: Africa, Asia, Europe, North America, South America
           Countries:  Argentina, Haiti, Iceland, India, Kenya, Liberia, Moldova, Montenegro, Mozambique, Portugal, Sierra Leone, Uganda, Ukraine
--------------------------------------------------------------------------------
Bucket 02: Saved 51 scenes (762 potential matches) -> student_02_scenes.gpkg
           Continents: Africa, Asia, Europe, North America, South America
           Countries:  Bangladesh, Belarus, Bolivia, Canada, Greece, Malawi, Mexico, Montenegro, Mozambique, Norway, Puerto Rico, Sierra Leone, Ukraine, United Kingdom, United Republic of Tanzania, United States of America
--------------------------------------------------------------------------------
Bucket 03: Saved 28 scenes (740

In [189]:
student_gdf

,_id,uuid,title,projection,bbox,footprint,gsd,file_size,acquisition_start,acquisition_end,...,area_km2,total_buildings,total_svi_points,buildings_on_path,valid_los_matches,has_overlap,country,continent,scene_id,bucket_id
10,60d204f1eade0f00079b58bf,https://oin-hotosm-temp.s3.amazonaws.com/60d20...,"Brgy. Pedro Vera Summit, Viga","PROJCS[""WGS 84 / UTM zone 51N"",GEOGCS[""WGS 84""...","[124.265884, 13.787232, 124.275683, 13.795791]","POLYGON ((124.265884 13.787232, 124.275683 13....",0.020000,224520893,2021-05-25T04:00:00.000Z,2021-05-26T03:59:00.000Z,...,1.0036,347,497,283,5,False,Philippines,Asia,60d204f1eade0f00079b58bf,10
20,6512962cceed5800019b01a7,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,Baie Nettle post Irma hurricane,"\nPROJCRS[""WGS 84 / UTM zone 20N"",\n BASEGE...","[-63.117192, 18.057778, -63.106167, 18.066083]","POLYGON ((-63.117192 18.057778, -63.106167 18....",0.022780,213966879,2017-11-02T23:00:00.000Z,2017-11-03T00:00:00.000Z,...,1.0726,295,1,4,1,False,Puerto Rico,North America,6512962cceed5800019b01a7,10
23,6a5ebe846f2574e69520d8e3,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,MARIANO ROQUE ALONSO - LITORAL SUR,"\nPROJCRS[""WGS 84 / UTM zone 21S"",\n BASEGE...","[-57.577114, -25.235386, -57.555704, -25.210137]","POLYGON ((-57.577114 -25.235386, -57.555704 -2...",0.023410,932289971,2026-01-15T03:00:00.000Z,2026-01-30T04:00:00.000Z,...,6.0255,4906,270,1966,18,False,Paraguay,South America,6a5ebe846f2574e69520d8e3,10
34,5d9977910ca8d70007c491d9,https://oin-hotosm-temp.s3.amazonaws.com/5d997...,Kallucata,"PROJCS[""WGS 84 / UTM zone 19S"",GEOGCS[""WGS 84""...","[-68.306035, -16.522368, -68.29568, -16.510509]","POLYGON ((-68.306035 -16.522368, -68.29568 -16...",0.025000,418366020,2018-11-24T14:00:00.000Z,2018-11-24T15:00:00.000Z,...,1.4506,12,1358,12,2,False,Bolivia,South America,5d9977910ca8d70007c491d9,10
42,59e62b963d6412ef7220a3fd,https://oin-hotosm-temp.s3.amazonaws.com/59639...,"Barangay Wilson, Mayorga, Leyte","PROJCS[""WGS 84 / UTM zone 51N"",GEOGCS[""WGS 84""...","[124.95670756098147, 10.843446771755657, 124.9...","POLYGON ((124.956707560981 10.8434467717557,12...",0.027130,661951246,2017-05-10T19:04:00.000Z,2017-05-16T16:25:00.000Z,...,3.9473,461,1343,299,12,True,Philippines,Asia,59e62b963d6412ef7220a3fd,10
44,617ea3af8ebcb9000515c52a,https://oin-hotosm-temp.s3.amazonaws.com/617e8...,chofu20211031disasterdrill0907mapconcierge00DSM,"PROJCS[""WGS 84 / UTM zone 54N"",GEOGCS[""WGS 84""...","[139.542207, 35.633591, 139.563791, 35.643102]","POLYGON ((139.542207 35.633591, 139.563791 35....",0.027470,4319280166,2021-10-31T00:00:00.000Z,2021-10-31T01:00:00.000Z,...,2.0556,2862,285,120,0,False,Japan,Asia,617ea3af8ebcb9000515c52a,10
45,617e93268ebcb9000515c509,https://oin-hotosm-temp.s3.amazonaws.com/617e9...,chofu20211031disasterdrill0907mapconcierge00,"PROJCS[""WGS 84 / UTM zone 54N"",GEOGCS[""WGS 84""...","[139.542207, 35.633591, 139.563791, 35.643102]","POLYGON ((139.542207 35.633591, 139.563791 35....",0.027470,275951166,2021-10-31T00:00:00.000Z,2021-10-31T01:00:00.000Z,...,2.0556,3055,162,56,0,False,Japan,Asia,617e93268ebcb9000515c509,10
58,5a95bd405a9ef7cb5d8e1a40,https://oin-hotosm-temp.s3.amazonaws.com/5a95b...,Laplaine - North section - Dominica,"PROJCS[""WGS 84 / UTM zone 20N"",GEOGCS[""WGS 84""...","[-61.254509417583115, 15.332117276173879, -61....","POLYGON ((-61.2545094175831 15.3321172761739,-...",0.029320,130210488,2018-02-16T08:00:00.000Z,2018-02-16T09:00:00.000Z,...,1.1900,138,270,28,1,True,Puerto Rico,North America,5a95bd405a9ef7cb5d8e1a40,10
59,617f0bde8ebcb9000515c52e,https://oin-hotosm-temp.s3.amazonaws.com/617f0...,chofu20211031,"PROJCS[""WGS 84 / UTM zone 54N"",GEOGCS[""WGS 84""...","[139.541335, 35.631844, 139.565628, 35.644555]","POLYGON ((139.541335 35.631844, 139.565628 35....",0.029497,258817971,2021-10-30T21:00:00.000Z,2021-10-31T00:00:00.000Z,...,3.0920,5860,144,43,1,False,Japan,Asia,617f0bde8ebcb9000515c52e,10
74,6633bbde6049ef00013b8250,https://oin-hotosm-temp.s3.us-east-1